# TCGA-BRCA Inventory Review

This notebook is review-only. It loads the latest saved TCGA-BRCA inventory outputs from disk, checks basic inventory counts, and writes summary tables for human audit.

It does not call the GDC API, download files, build a cohort, choose an endpoint, or make clinical claims.

## Load the latest saved inventory snapshot

This section confirms that the stable manifest exists and points to a completed inventory run.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the current working directory.")


def parse_json_array_cell(value: object) -> list[str]:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    text = str(value).strip()
    if not text:
        return []
    parsed = json.loads(text)
    if not isinstance(parsed, list):
        raise ValueError(f"Expected a JSON array cell, received: {text}")
    return [str(item).strip() for item in parsed if str(item).strip()]


repo_root = find_repo_root(Path.cwd())
latest_manifest_path = repo_root / "01-data" / "audit" / "tcga-brca" / "source" / "tcga_brca_inventory_latest.json"
if not latest_manifest_path.exists():
    raise FileNotFoundError(
        f"Latest inventory manifest not found: {latest_manifest_path}. Run the fetch script first."
    )

latest_manifest = json.loads(latest_manifest_path.read_text(encoding="utf-8"))
case_inventory_path = repo_root / latest_manifest["case_inventory_tsv"]
file_inventory_path = repo_root / latest_manifest["file_inventory_tsv"]
run_log_path = repo_root / latest_manifest["run_log_json"]
results_root = repo_root / "09-trials" / "01-tcga-only-source-audited" / "05-results"
results_root.mkdir(parents=True, exist_ok=True)

display(pd.DataFrame([latest_manifest]))

,updated_at_utc,run_id,gdc_data_release,gdc_tag,run_directory,case_inventory_tsv,file_inventory_tsv,run_log_json
0,2026-04-11T22:50:36Z,20260411T224824Z,"Data Release 45.0 - December 04, 2025",8.3.1,01-data/audit/tcga-brca/source/runs/20260411T2...,01-data/audit/tcga-brca/source/runs/20260411T2...,01-data/audit/tcga-brca/source/runs/20260411T2...,01-data/audit/tcga-brca/source/runs/20260411T2...


## Load the saved TSV inventories

This section reads the normalized case and file inventories from disk and confirms that the core row counts are available for review.

In [2]:
case_df = pd.read_csv(case_inventory_path, sep="\t")
file_df = pd.read_csv(file_inventory_path, sep="\t")

print(f"Case inventory path: {case_inventory_path}")
print(f"File inventory path: {file_inventory_path}")
print(f"Run log path: {run_log_path}")
print(f"Cases loaded: {len(case_df):,}")
print(f"Files loaded: {len(file_df):,}")

Case inventory path: d:\Projects\brcapath-rx\01-data\audit\tcga-brca\source\runs\20260411T224824Z\tcga_brca_case_inventory.tsv
File inventory path: d:\Projects\brcapath-rx\01-data\audit\tcga-brca\source\runs\20260411T224824Z\tcga_brca_file_inventory.tsv
Run log path: d:\Projects\brcapath-rx\01-data\audit\tcga-brca\source\runs\20260411T224824Z\run_log.json
Cases loaded: 1,098
Files loaded: 70,774


## Case-level inventory checks

This section checks the basic size of the case inventory and saves a compact case summary table for the trial results folder.

In [3]:
case_summary = pd.DataFrame(
    [
        {
            "metric": "case_count",
            "value": int(len(case_df)),
        },
        {
            "metric": "unique_case_submitter_count",
            "value": int(case_df["submitter_id"].nunique(dropna=True)),
        },
    ]
)
case_summary_path = results_root / "01_case_inventory_summary.tsv"
case_summary.to_csv(case_summary_path, sep="\t", index=False)
display(case_summary)

,metric,value
0,case_count,1098
1,unique_case_submitter_count,1098


## File-level categorical summaries

These tables summarize what kinds of TCGA-BRCA files are currently indexed in GDC. They are intended for availability review only.

In [4]:
def save_count_table(df: pd.DataFrame, column_name: str, output_name: str) -> pd.DataFrame:
    counts = (
        df[column_name]
        .fillna("<missing>")
        .astype(str)
        .value_counts(dropna=False)
        .rename_axis(column_name)
        .reset_index(name="file_count")
    )
    counts.to_csv(results_root / output_name, sep="\t", index=False)
    return counts


data_category_counts = save_count_table(file_df, "data_category", "02_file_counts_by_data_category.tsv")
data_type_counts = save_count_table(file_df, "data_type", "03_file_counts_by_data_type.tsv")
experimental_strategy_counts = save_count_table(
    file_df, "experimental_strategy", "04_file_counts_by_experimental_strategy.tsv"
)
access_counts = save_count_table(file_df, "access", "05_file_counts_by_access.tsv")

print("Top data categories")
display(data_category_counts.head(10))
print("Top data types")
display(data_type_counts.head(10))
print("Experimental strategies")
display(experimental_strategy_counts.head(10))
print("Access counts")
display(access_counts.head(10))

Top data categories


,data_category,file_count
0,Simple Nucleotide Variation,21132
1,Copy Number Variation,14346
2,Sequencing Reads,9282
3,Structural Variation,5772
4,Biospecimen,5317
5,Transcriptome Profiling,4876
6,DNA Methylation,3714
7,Somatic Structural Variation,3128
8,Clinical,2288
9,Proteome Profiling,919


Top data types


,data_type,file_count
0,Annotated Somatic Mutation,10134
1,Aligned Reads,9282
2,Raw Simple Somatic Mutation,6751
3,Transcript Fusion,4924
4,Structural Rearrangement,3976
5,Gene Level Copy Number,3314
6,Copy Number Segment,3256
7,Slide Image,3112
8,Masked Intensities,2476
9,Raw Intensities,2263


Experimental strategies


,experimental_strategy,file_count
0,WXS,17049
1,Genotyping Array,14329
2,WGS,12383
3,RNA-Seq,11079
4,<missing>,4493
5,Methylation Array,3714
6,miRNA-Seq,3621
7,Tissue Slide,1979
8,Diagnostic Slide,1133
9,Reverse Phase Protein Array,919


Access counts


,access,file_count
0,controlled,42843
1,open,27931


## Sample-type and workflow review

The file inventory stores linked sample fields as JSON arrays in TSV cells. This section explodes those arrays for review-oriented counts and saves the outputs for the results folder.

In [5]:
if "sample_types" in file_df.columns:
    exploded_sample_types = file_df["sample_types"].apply(parse_json_array_cell).explode()
    sample_type_counts = (
        exploded_sample_types.dropna()
        .astype(str)
        .str.strip()
        .loc[lambda series: series != ""]
        .value_counts()
        .rename_axis("sample_type")
        .reset_index(name="linked_file_count")
    )
else:
    sample_type_counts = pd.DataFrame(columns=["sample_type", "linked_file_count"])
sample_type_counts.to_csv(results_root / "06_file_counts_by_sample_type.tsv", sep="\t", index=False)

workflow_series = file_df["analysis_workflow_type"].fillna("").astype(str).str.strip()
workflow_series = workflow_series.loc[workflow_series != ""]
if workflow_series.empty:
    workflow_type_counts = pd.DataFrame(columns=["analysis_workflow_type", "file_count"])
else:
    workflow_type_counts = (
        workflow_series.value_counts()
        .rename_axis("analysis_workflow_type")
        .reset_index(name="file_count")
    )
workflow_type_counts.to_csv(results_root / "07_file_counts_by_workflow_type.tsv", sep="\t", index=False)

print("Sample types")
display(sample_type_counts.head(10))
print("Workflow types")
display(workflow_type_counts.head(10))

Sample types


,sample_type,linked_file_count
0,Primary Tumor,58244
1,Blood Derived Normal,33721
2,Solid Tissue Normal,4525
3,Metastatic,309


Workflow types


,analysis_workflow_type,file_count
0,DNAcopy,4458
1,BWA with Mark Duplicates and BQSR,4382
2,SeSAMe Methylation Beta Estimation,3714
3,VarScan2 Annotation,2986
4,Arriba,2462
5,STAR - Counts,2462
6,STAR-Fusion,2462
7,BCGSC miRNA Profiling,2414
8,Birdseed,2263
9,ASCAT2,2168


## Review reminder

These outputs support source inventory review only. Human interpretation is still required before choosing download targets, cohort rules, endpoints, or any downstream analysis.